In [11]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

In [12]:
df = pd.read_csv("../data/processed/credit_data_clean.csv")

df.head()

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,...,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk,default
0,... < 100 DM,6,critical account/other credits existing,domestic appliances,1169,unknown/no savings account,... >= 7 years,4,male : single,none,...,67,none,own,2,skilled employee/official,1,yes,yes,1,1
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,domestic appliances,5951,... < 100 DM,1 <= ... < 4 years,2,female : divorced/separated/married,none,...,22,none,own,1,skilled employee/official,1,no,yes,0,0
2,no checking account,12,critical account/other credits existing,retraining,2096,... < 100 DM,4 <= ... < 7 years,2,male : single,none,...,49,none,own,1,unskilled - resident,2,no,yes,1,1
3,... < 100 DM,42,existing credits paid back duly till now,radio/television,7882,... < 100 DM,4 <= ... < 7 years,2,male : single,guarantor,...,45,none,for free,1,skilled employee/official,2,no,yes,1,1
4,... < 100 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : single,none,...,53,none,for free,2,skilled employee/official,2,no,yes,0,0


In [13]:
selected_cols = [
    "status",
    "credit_history",
    "purpose",
    "savings",
    "employment_duration",
    "age",
    "duration",
    "amount",
    "installment_rate",
    "number_credits",
    "housing",
    "telephone",
    "foreign_worker",
    "personal_status_sex"
]

X = df[selected_cols].copy()
y = df["default"]

In [14]:
X["amount_duration_ratio"] = X["amount"] / X["duration"]
X["log_amount"] = np.log1p(X["amount"])
X["duration_squared"] = X["duration"] ** 2
X["young_flag"] = (X["age"] < 25).astype(int)
X["senior_flag"] = (X["age"] > 60).astype(int)
X["multiple_credits"] = (X["number_credits"] > 1).astype(int)
X["long_duration"] = (X["duration"] > 36).astype(int)
X["high_installment"] = (X["installment_rate"] >= 3).astype(int)

X.head()

,status,credit_history,purpose,savings,employment_duration,age,duration,amount,installment_rate,number_credits,...,foreign_worker,personal_status_sex,amount_duration_ratio,log_amount,duration_squared,young_flag,senior_flag,multiple_credits,long_duration,high_installment
0,... < 100 DM,critical account/other credits existing,domestic appliances,unknown/no savings account,... >= 7 years,67,6,1169,4,2,...,yes,male : single,194.833333,7.064759,36,0,1,1,0,1
1,0 <= ... < 200 DM,existing credits paid back duly till now,domestic appliances,... < 100 DM,1 <= ... < 4 years,22,48,5951,2,1,...,yes,female : divorced/separated/married,123.979167,8.691483,2304,1,0,0,1,0
2,no checking account,critical account/other credits existing,retraining,... < 100 DM,4 <= ... < 7 years,49,12,2096,2,1,...,yes,male : single,174.666667,7.648263,144,0,0,0,0,0
3,... < 100 DM,existing credits paid back duly till now,radio/television,... < 100 DM,4 <= ... < 7 years,45,42,7882,2,1,...,yes,male : single,187.666667,8.972464,1764,0,0,0,1,0
4,... < 100 DM,delay in paying off in the past,car (new),... < 100 DM,1 <= ... < 4 years,53,24,4870,3,2,...,yes,male : single,202.916667,8.491055,576,0,0,1,0,1


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [16]:
categorical_cols = [
    "status",
    "credit_history",
    "purpose",
    "savings",
    "employment_duration",
    "housing",
    "telephone",
    "foreign_worker",
    "personal_status_sex"
]

numeric_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [17]:
log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000))
])

log_model.fit(X_train, y_train)

log_probs = log_model.predict_proba(X_test)[:, 1]
print("Logistic Regression AUC:", roc_auc_score(y_test, log_probs))

Logistic Regression AUC: 0.7917857142857143


In [18]:
lgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ))
])

lgb_model.fit(X_train, y_train)

lgb_probs = lgb_model.predict_proba(X_test)[:, 1]
print("LightGBM AUC:", roc_auc_score(y_test, lgb_probs))

[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Warning] Accuracy may be bad since you didn't explicitly set num_leaves OR 2^max_depth > num_leaves. (num_leaves=31).
[LightGBM] [Info] Number of positive: 560, number of negative: 240
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000295 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 960
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

In [19]:
joblib.dump(lgb_model, "../models/final_credit_model.pkl")

print("New strong model saved successfully!")

New strong model saved successfully!
